In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

games = pd.read_csv('/kaggle/input/nfl-big-data-bowl-2025/games.csv')
players = pd.read_csv('/kaggle/input/nfl-big-data-bowl-2025/players.csv')
player_play = pd.read_csv('/kaggle/input/nfl-big-data-bowl-2025/player_play.csv')
plays = pd.read_csv('/kaggle/input/nfl-big-data-bowl-2025/plays.csv')
week2 = pd.read_csv('/kaggle/input/nfl-big-data-bowl-2025/tracking_week_2.csv')

/kaggle/input/nfl-big-data-bowl-2025/players.csv
/kaggle/input/nfl-big-data-bowl-2025/tracking_week_7.csv
/kaggle/input/nfl-big-data-bowl-2025/tracking_week_9.csv
/kaggle/input/nfl-big-data-bowl-2025/tracking_week_6.csv
/kaggle/input/nfl-big-data-bowl-2025/games.csv
/kaggle/input/nfl-big-data-bowl-2025/tracking_week_8.csv
/kaggle/input/nfl-big-data-bowl-2025/player_play.csv
/kaggle/input/nfl-big-data-bowl-2025/tracking_week_4.csv
/kaggle/input/nfl-big-data-bowl-2025/tracking_week_3.csv
/kaggle/input/nfl-big-data-bowl-2025/tracking_week_5.csv
/kaggle/input/nfl-big-data-bowl-2025/tracking_week_1.csv
/kaggle/input/nfl-big-data-bowl-2025/plays.csv
/kaggle/input/nfl-big-data-bowl-2025/tracking_week_2.csv


In [2]:
week2 = week2.merge(players[['nflId', 'position']], on='nflId', how='left')

# Step 1: Get the reference values (value_x, value_y) for each (gameId, playId)
reference_values = week2[week2['frameId'] == 1 & (week2['club'] == 'football')] \
    .groupby(['gameId', 'playId'])[['x', 'y']].first().reset_index()

# Step 2: Merge the reference values back to the original dataframe
week2 = week2.merge(reference_values, on=['gameId', 'playId'], suffixes=('', '_ref'))

# Step 3: Adjust the x and y coordinates by subtracting the reference values
week2['x'] = week2['x'] - week2['x_ref']
week2['y'] = week2['y'] - week2['y_ref']

# Step 4: Drop the temporary reference columns
week2 = week2.drop(columns=['x_ref', 'y_ref'])
week2[week2['club'] == 'DET']

,gameId,playId,nflId,displayName,frameId,frameType,time,jerseyNumber,club,playDirection,x,y,s,a,dis,o,dir,event,position
4936581,2022091802,56,43290.0,Jared Goff,1,BEFORE_SNAP,2022-09-18 17:03:18.3,16.0,DET,right,-6.010001,0.430001,0.52,0.99,0.05,127.60,135.70,huddle_break_offense,QB
4936582,2022091802,56,43290.0,Jared Goff,2,BEFORE_SNAP,2022-09-18 17:03:18.4,16.0,DET,right,-5.970001,0.390001,0.59,0.77,0.06,123.70,135.51,NaN,QB
4936583,2022091802,56,43290.0,Jared Goff,3,BEFORE_SNAP,2022-09-18 17:03:18.5,16.0,DET,right,-5.930001,0.340001,0.66,0.49,0.06,119.42,133.79,NaN,QB
4936584,2022091802,56,43290.0,Jared Goff,4,BEFORE_SNAP,2022-09-18 17:03:18.6,16.0,DET,right,-5.880001,0.300001,0.68,0.22,0.07,118.43,131.05,NaN,QB
4936585,2022091802,56,43290.0,Jared Goff,5,BEFORE_SNAP,2022-09-18 17:03:18.7,16.0,DET,right,-5.820001,0.260001,0.69,0.10,0.07,116.81,126.20,NaN,QB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5407012,2022091802,3980,53959.0,Brock Wright,57,AFTER_SNAP,2022-09-18 20:07:41.4,89.0,DET,left,1.630002,3.390001,0.43,0.09,0.07,257.21,193.97,qb_kneel,TE
5407013,2022091802,3980,53959.0,Brock Wright,58,AFTER_SNAP,2022-09-18 20:07:41.5,89.0,DET,left,1.610002,3.350001,0.41,0.14,0.04,258.35,196.88,NaN,TE
5407014,2022091802,3980,53959.0,Brock Wright,59,AFTER_SNAP,2022-09-18 20:07:41.6,89.0,DET,left,1.590002,3.320001,0.38,0.18,0.03,259.92,199.27,NaN,TE
5407015,2022091802,3980,53959.0,Brock Wright,60,AFTER_SNAP,2022-09-18 20:07:41.7,89.0,DET,left,1.580002,3.310001,0.33,0.19,0.02,261.45,202.66,NaN,TE


In [3]:
# Step 2: Find the 'line_set' and 'ball_snap' event indices for each player in each play
def get_event_range(group):
    # Check if both 'line_set' and 'ball_snap' events exist in the group
    if 'line_set' in group['event'].values and 'ball_snap' in group['event'].values:
        # Get the first occurrence of 'line_set' and 'ball_snap'
        lineset_index = group[group['event'] == 'line_set'].index[0]  
        ballsnap_index = group[group['event'] == 'ball_snap'].index[0]  

        # Extract the slice of the DataFrame from 'line_set' to 'ball_snap'
        return group.loc[lineset_index:ballsnap_index]
    else:
        # If either event is missing, return an empty DataFrame to skip that group
        return pd.DataFrame()  # Return an empty DataFrame

# Step 3: Group by 'playId' and 'nflId', apply the event range extraction, and concatenate the results
week2 = week2.groupby(['playId', 'nflId']).apply(get_event_range)

# Step 4: Reset the index to remove multi-level index after groupby
week2 = week2.reset_index(drop=True)

# Step 5: Display the result
week2[week2['club'] == 'DET']

,gameId,playId,nflId,displayName,frameId,frameType,time,jerseyNumber,club,playDirection,x,y,s,a,dis,o,dir,event,position
3695,2.022092e+09,56.0,43290.0,Jared Goff,35.0,BEFORE_SNAP,2022-09-18 17:03:21.7,16.0,DET,right,-4.220001,-0.049999,0.60,0.45,0.06,88.79,66.24,line_set,QB
3696,2.022092e+09,56.0,43290.0,Jared Goff,36.0,BEFORE_SNAP,2022-09-18 17:03:21.8,16.0,DET,right,-4.170001,-0.019999,0.56,0.56,0.06,87.08,59.19,NaN,QB
3697,2.022092e+09,56.0,43290.0,Jared Goff,37.0,BEFORE_SNAP,2022-09-18 17:03:21.9,16.0,DET,right,-4.140001,0.010001,0.44,0.77,0.05,86.24,51.56,NaN,QB
3698,2.022092e+09,56.0,43290.0,Jared Goff,38.0,BEFORE_SNAP,2022-09-18 17:03:22,16.0,DET,right,-4.110001,0.030001,0.27,1.03,0.03,88.09,56.46,NaN,QB
3699,2.022092e+09,56.0,43290.0,Jared Goff,39.0,BEFORE_SNAP,2022-09-18 17:03:22.1,16.0,DET,right,-4.090001,0.040001,0.13,1.11,0.02,88.09,53.31,NaN,QB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2232340,2.022092e+09,3980.0,53959.0,Brock Wright,46.0,BEFORE_SNAP,2022-09-18 20:07:40.3,89.0,DET,left,1.660002,3.650001,0.05,0.02,0.01,253.59,280.48,NaN,TE
2232341,2.022092e+09,3980.0,53959.0,Brock Wright,47.0,BEFORE_SNAP,2022-09-18 20:07:40.4,89.0,DET,left,1.650002,3.650001,0.05,0.08,0.01,253.59,272.90,NaN,TE
2232342,2.022092e+09,3980.0,53959.0,Brock Wright,48.0,BEFORE_SNAP,2022-09-18 20:07:40.5,89.0,DET,left,1.650002,3.640001,0.08,0.47,0.01,253.59,195.68,NaN,TE
2232343,2.022092e+09,3980.0,53959.0,Brock Wright,49.0,BEFORE_SNAP,2022-09-18 20:07:40.6,89.0,DET,left,1.650002,3.630001,0.15,0.68,0.01,253.59,178.32,NaN,TE


In [4]:
week2.loc[week2['playDirection'] == 'left', 'x'] = week2.loc[week2['playDirection'] == 'left', 'x'] * -1
week2.loc[week2['playDirection'] == 'left', 'y'] = week2.loc[week2['playDirection'] == 'left', 'y'] * -1
week2.loc[week2['playDirection'] == 'left', 'o'] = week2.loc[week2['playDirection'] == 'left', 'o'].apply(lambda o: 360 - o)

week2['gameId'] = week2['gameId'].astype(int)
week2['playId'] = week2['playId'].astype(int)
week2['nflId'] = week2['nflId'].astype(int)
week2['frameId'] = week2['frameId'].astype(int)

In [5]:
test = week2[(week2['club'] == 'DET') & (week2['position'].isin(['QB', 'RB', 'WR', 'TE', 'G', 'T', 'C']))]

In [6]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

fig_dict = {
    'data': [],
    'layout': {},
    'frames': []
}

fig_dict['layout']['xaxis'] = {
    'range': [-20, 20],
    'title': {'text': 'x', 'font': {'color': 'white'}},  
    'linecolor': 'black',  
    'tickcolor': 'black',  
    'dtick': 10, 
    'showgrid': True 
}

fig_dict['layout']['yaxis'] = {
    'range': [-25, 25],
    'title': {'text': 'y', 'font': {'color': 'white'}},
    'linecolor': 'black', 
    'tickcolor': 'black',  
    'dtick': 10, 
    'showgrid': False  
}

fig_dict['layout']['hovermode'] = 'closest'
fig_dict['layout']['font'] = {'color': 'white'} 
fig_dict['layout']['updatemenus'] = [
    {
        "buttons": [
            {
                'args': [None, {'frame': {'duration': 50, 'redraw': False},
                                'fromcurrent': True, 'transition': {'duration': 1000}}],
                'label': 'Play',
                'method': 'animate'
            },
            {
                'args': [[None], {'frame': {'duration': 0, 'redraw': False},
                                  'mode': 'immediate',
                                  'transition': {'duration': 0}}],
                'label': 'Pause',
                'method': 'animate'
            }
        ],
        'direction': 'left',
        'pad': {'r': 10, 't': 87},
        'showactive': False,
        'type': 'buttons',
        'x': 0.1,
        'xanchor': 'right',
        'y': 0,
        'yanchor': 'top'
    }
]


sliders_dict = {
    'active': 0,
    'yanchor': 'top',
    'xanchor': 'left',
    'currentvalue': {
        'font': {'size': 20, 'color': 'black'},
        'prefix': 'Time:',
        'visible': True,
        'xanchor': 'right'
    },
    'transition': {'duration': 300},
    'pad': {'b': 10, 't': 50},
    'len': .9,
    'x': .1,
    'y': 0,
    'steps': []
}

for club in test['club'].unique():
    data_dict = {
        'x': test.loc[test['club'] == club, 'x'],
        'y': test.loc[test['club'] == club, 'y'],
        'mode': 'markers',
        'text': test.loc[test['club'] == club, 'displayName'],
        'name': club
    }
    fig_dict['data'].append(data_dict)

initial_frame = {'data': [], 'name': 'Initial'}
fig_dict['frames'].append(initial_frame)

for frame in test['frameId'].unique():
    f = {'data': [], 'name': str(frame)}
    
    for club in test['club'].unique():
        frame_data = {
            'x': test.loc[test['frameId'] == frame, 'x'][test['club'] == club],
            'y': test.loc[test['frameId'] == frame, 'y'][test['club'] == club],
            'mode': 'markers',
            'text': test.loc[test['frameId'] == frame, 'displayName'][test['club'] == club],
            'name': club
        }
        f['data'].append(frame_data) 

    fig_dict['frames'].append(f)
    
    slider_step = {
        'args': [
            [frame],
            {'frame': {'duration': 0, 'redraw': False},
             'mode': 'immediate',
             'transition': {'duration': 0}}
        ],
        'label': str(frame),
        'method': 'animate'
    }
    sliders_dict['steps'].append(slider_step) 

fig_dict['layout']['sliders'] = [sliders_dict]

fig = go.Figure(fig_dict)

if len(fig_dict['frames']) > 1:
    fig.update(data=fig_dict['frames'][1]['data']) 

fig.update_layout(paper_bgcolor='#242424', plot_bgcolor='#0a4200')
    
fig.show()